In [1]:
import os
import sys
sys.path.append(r'X:\Exchange\Guillermo\analysis\flowshape\flowshape_pipeline')

data_folder = r'X:\mcdole\KM_16-02-03\analysis\000_flowshape\pre_mesh'
postmesh_folder = r'X:\mcdole\KM_16-02-03\analysis\001_flowshape\post_mesh'
results_folder = r'X:\mcdole\KM_16-02-03\analysis\001_flowshape'

skip = 1
ti = 0
tf = 151

os.makedirs(postmesh_folder, exist_ok=True)
os.makedirs(results_folder, exist_ok=True)


In [2]:
from helper_functions import load_dataset

data = load_dataset(data_folder, ti=ti, tf=tf, skip=skip)
len(data)

Loading OBJs: 100%|██████████| 151/151 [01:45<00:00,  1.43it/s]


151

In [3]:
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
from mesh_functions import heal_single_mesh

os.makedirs(postmesh_folder, exist_ok=True)

jobs = [
    delayed(heal_single_mesh)(
        d["v"], d["f"],
        output_path=os.path.join(postmesh_folder, f"{i:04d}.obj"),
    )
    for i, d in enumerate(data)
]

healed = Parallel(n_jobs=-1, return_as="generator")(jobs)
healed = list(tqdm(healed, total=len(jobs), desc="Healing meshes"))

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


Healing meshes:   0%|          | 0/151 [00:00<?, ?it/s]

In [4]:
import importlib
import mesh_functions
importlib.reload(mesh_functions)
from mesh_functions import interactive_rotation_widget

time_point = -1
rot_state = interactive_rotation_widget(healed[time_point]["v"], healed[time_point]["f"])

Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, children=(DirectionalLight(color='white', intensi…

In [ ]:
from mesh_functions import interactive_mask_plane_widget

plane_state = interactive_mask_plane_widget(
    healed[time_point]["v"], healed[time_point]["f"],
    T_rotation=rot_state["T_rotation"],
    results_folder=results_folder,
)

Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, children=(DirectionalLight(color='white', intensi…

In [8]:
importlib.reload(mesh_functions)
from mesh_functions import compute_face_mask

f_mask = compute_face_mask(
    healed[time_point]["v"], healed[time_point]["f"],
    transform=rot_state["T_rotation"],
    mask_plane_normal=plane_state["mask_plane_normal"],
    mask_plane_offset=plane_state["mask_plane_offset"],
    mask_side=plane_state["mask_side"],
    plot=True,
)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(595.57127…

In [9]:
from mesh_functions import transform_single_mesh

T_rot = rot_state["T_rotation"]
normal = plane_state["mask_plane_normal"]
offset = plane_state["mask_plane_offset"]
side = plane_state["mask_side"]

jobs = [
    delayed(transform_single_mesh)(
        d["v"], d["f"],
        transform=T_rot,
        mask_plane_normal=normal,
        mask_plane_offset=offset,
        mask_side=side,
    )
    for d in healed
]

transformed = Parallel(n_jobs=-1, return_as="generator")(jobs)
transformed = list(tqdm(transformed, total=len(jobs), desc="Rotating & masking"))

Rotating & masking:   0%|          | 0/151 [00:00<?, ?it/s]

In [10]:
import importlib
from helper_functions import build_gaussian, compute_harmonics_single


smoothing_mask = build_gaussian(k=0.01, l_max=16)

jobs = [
    delayed(compute_harmonics_single)(
        d["v"], d["f"], d["f_mask"], smoothing_mask,
    )
    for d in transformed
]

time_points = Parallel(n_jobs=-1, return_as="generator")(jobs)
time_points = list(tqdm(time_points, total=len(jobs), desc="Computing harmonics"))

Computing harmonics:   0%|          | 0/151 [00:00<?, ?it/s]

In [11]:
import importlib
from helper_functions import save_results

save_results(
    os.path.join(results_folder, "results.h5"),
    time_points,
    params={
        "T_rotation": rot_state["T_rotation"],
        "mask_plane_normal": plane_state["mask_plane_normal"],
        "mask_plane_offset": plane_state["mask_plane_offset"],
        "mask_side": plane_state["mask_side"],
    },
)

Saving HDF5:   0%|          | 0/151 [00:00<?, ?it/s]

Saving HDF5: 100%|██████████| 151/151 [00:34<00:00,  4.39it/s]


In [14]:
import importlib
import helper_functions
importlib.reload(helper_functions)
from helper_functions import load_results

tp = load_results(
    os.path.join(results_folder, "results.h5"),
    keys=["v", "f", "Y_mat", "weights_masked", "computed"],
)

Loading HDF5: 100%|██████████| 151/151 [00:26<00:00,  5.61it/s]


In [16]:
import meshplot
import numpy as np

i = 21
assert tp[i].get("computed") == 1, f"Time point {i} was not computed successfully"

v = tp[i]["v"]
f = tp[i]["f"]
c = tp[i]["Y_mat"] @ tp[i]["weights_masked"]

centroid = v.mean(axis=0)
v_c = v - centroid
extent = np.linalg.norm(v_c.max(axis=0) - v_c.min(axis=0))
print(f"Extent: {extent:.1f}")

p = meshplot.plot(v_c, f, c=c, shading={"width": 1200, "height": 600})
p._renderer.camera.position = [0, 0, 2000]

Extent: 1087.6


Renderer(camera=PerspectiveCamera(aspect=2.0, children=(DirectionalLight(color='white', intensity=0.6, positio…

In [18]:
import importlib
import mesh_functions
importlib.reload(mesh_functions)
from mesh_functions import interactive_render_preview

t = time_points[75]
v_c = t["v"] - t["v"].mean(axis=0)
c = t["Y_mat"] @ t["weights_masked"]
clim_vals = []
for tp in time_points:
    if tp.get("computed") == 1:
        vals = tp["Y_mat"] @ tp["weights_masked"]
        clim_vals.extend([vals.min(), vals.max()])
clim = (min(clim_vals), max(clim_vals))

view_state = interactive_render_preview(v_c, t["f"], scalars=c, clim=clim)

In [19]:
import importlib
import mesh_functions
importlib.reload(mesh_functions)
from mesh_functions import render_single_frame, make_movie

frames_folder = os.path.join(results_folder, "movie_frames")
os.makedirs(frames_folder, exist_ok=True)
print(view_state)
# Build jobs for parallel rendering
jobs = []
for i, t in enumerate(time_points):
    if t.get("computed") != 1:
        continue
    v = t["v"]
    v_c = v - v.mean(axis=0)
    c = t["Y_mat"] @ t["weights_masked"]
    jobs.append(
        delayed(render_single_frame)(
            v_c, t["f"],
            output_path=os.path.join(frames_folder, f"{i:04d}.png"),
            scalars=c,
            clim=clim,
            elev=view_state["elev"],
            azim=view_state["azim"],
        )
    )

rendered = Parallel(n_jobs=-1, return_as="generator")(jobs)
list(tqdm(rendered, total=len(jobs), desc="Rendering frames"))

# Stitch into video
movie_path = os.path.join(results_folder, "curvature_movie.mp4")
make_movie(frames_folder, movie_path, fps=15)
print(f"Movie saved to {movie_path}")

{'elev': 70, 'azim': -90}


Rendering frames:   0%|          | 0/148 [00:00<?, ?it/s]

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (924, 924) to (928, 928) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Movie saved to X:\mcdole\KM_16-02-03\analysis\001_flowshape\curvature_movie.mp4
